# Laboratorio 6 — Red Neuronal Convolucional (CNN) sobre MNIST

**Inteligencia Artificial: Aprendizaje Automático — UCAB**

Autores:
- Jesús Gil — cédula `30175126`
- Gabriel Castellano — cédula `28059781`

## Objetivo

Diseñar e implementar **desde cero** un modelo basado en redes neuronales convolucionales (CNN) que clasifique los dígitos del conjunto de datos **MNIST**. El modelo debe ser competitivo con los modelos vistos en clase. Las medidas de rendimiento reportadas durante el entrenamiento y la validación son **accuracy** y **AUC**.

Se sigue el mismo estilo y patrones de los notebooks de referencia del laboratorio (uso de `tf.keras.layers.Conv2D` con `partial`, inicialización `he_normal`, optimizador `nadam`, arquitectura tipo VGG reducida). La única adaptación es el dataset (MNIST de dígitos en lugar de Fashion MNIST) y la incorporación explícita de la métrica AUC.

## 1. Importación de librerías

Se importan TensorFlow/Keras (construcción y entrenamiento del modelo), NumPy y Matplotlib (manipulación y visualización) y utilidades de scikit-learn para complementar el reporte.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from functools import partial
from sklearn.metrics import confusion_matrix, classification_report

print('TensorFlow:', tf.__version__)
print('Keras    :', tf.keras.__version__)
print('GPU disponible:', bool(tf.config.list_physical_devices('GPU')))

plt.rc('font', size=12)
plt.rc('image', cmap='gray')
plt.rc('figure', autolayout=True)

# Reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

IS_COLAB = 'google.colab' in sys.modules
print('Ejecutando en Colab:', IS_COLAB)

## 2. Carga del conjunto de datos MNIST

MNIST contiene 70 000 imágenes de dígitos manuscritos (0–9) en escala de grises, de 28×28 píxeles. Keras ya provee el dataset particionado en 60 000 imágenes de entrenamiento y 10 000 de prueba.

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print('Train completo:', X_train_full.shape, y_train_full.shape)
print('Test         :', X_test.shape, y_test.shape)
print('Rango pixeles:', X_train_full.min(), '..', X_train_full.max())
print('Clases       :', np.unique(y_train_full))

Una muestra del dataset para confirmar visualmente las imágenes y sus etiquetas.

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train_full[i])
    plt.title(f'y = {y_train_full[i]}')
    plt.axis('off')
plt.suptitle('Muestra de MNIST')
plt.show()

## 3. Preprocesamiento

Se aplican las mismas transformaciones que utiliza el notebook de referencia del laboratorio (`Laboratorio_de_CNN.ipynb`, celda de Fashion MNIST):

1. **Expansión del canal**: las imágenes son 2D `(28, 28)`. Para una `Conv2D` necesitamos un tensor 4D `(n, 28, 28, 1)` (alto, ancho, canales).
2. **Normalización**: dividir por 255 para que los píxeles queden en el rango `[0, 1]`. Esto estabiliza el entrenamiento.
3. **Partición train / valid**: separamos los últimos 5 000 ejemplos del conjunto de entrenamiento para validación. Esto sigue el mismo criterio que el notebook del laboratorio.
4. **One-hot de las etiquetas**: para poder usar la métrica `AUC` de Keras con un problema multiclase necesitamos las etiquetas en formato one-hot y la pérdida `categorical_crossentropy`.

In [ ]:
# Expandir canal y normalizar a [0, 1]
X_train_full = np.expand_dims(X_train_full, axis=-1).astype(np.float32) / 255.0
X_test       = np.expand_dims(X_test,       axis=-1).astype(np.float32) / 255.0

# Particionar train/valid (mismas 5000 imágenes finales que en el notebook del Lab)
X_train, X_valid = X_train_full[:-5000], X_train_full[-5000:]
y_train, y_valid = y_train_full[:-5000], y_train_full[-5000:]

# One-hot para usar AUC en Keras
N_CLASES = 10
y_train_oh = tf.keras.utils.to_categorical(y_train, N_CLASES)
y_valid_oh = tf.keras.utils.to_categorical(y_valid, N_CLASES)
y_test_oh  = tf.keras.utils.to_categorical(y_test,  N_CLASES)

print('X_train:', X_train.shape, '  X_valid:', X_valid.shape, '  X_test:', X_test.shape)
print('y_train one-hot:', y_train_oh.shape)

## 4. Diseño y justificación del modelo

Tomamos como base la arquitectura tipo VGG reducida usada en el notebook de referencia del laboratorio (`Laboratorio_de_CNN.ipynb`, celda 60). Es una arquitectura **competitiva** sobre MNIST sin ser excesivamente grande para el tamaño del dataset.

### Bloques

| Etapa | Capas | Filtros | Justificación |
|---|---|---|---|
| 1 | `Conv2D(7×7) → MaxPool` | 64 | Una primera capa con kernel grande captura trazos amplios del dígito. |
| 2 | `Conv2D(3×3) × 2 → MaxPool` | 128 | Doble convolución profundiza la extracción de bordes y curvas. |
| 3 | `Conv2D(3×3) × 2 → MaxPool` | 256 | Mayor número de filtros para capturar combinaciones más abstractas. |
| 4 | `Flatten → Dense(128) + Dropout → Dense(64) + Dropout → Dense(10, softmax)` | — | Cabezal totalmente conectado con `Dropout 0.5` para reducir sobreajuste y `softmax` para producir una distribución de probabilidad sobre las 10 clases. |

### Decisiones de diseño

- **`padding='same'`** en todas las convoluciones para que el tamaño espacial sólo se reduzca con `MaxPool`. Esto facilita el seguimiento de dimensiones a través de la red.
- **Activación `ReLU`** en todas las capas ocultas (mismo criterio que el lab).
- **Inicialización `he_normal`**, adecuada para `ReLU` (mismo criterio que el lab).
- **`Dropout(0.5)`** en el cabezal denso para combatir el sobreajuste sobre un dataset pequeño en términos modernos (60 000 imágenes).
- **Salida `softmax`** de 10 unidades para obtener probabilidades por clase.
- **Optimizador `Nadam`**, una mejora de Adam con momento Nesterov; el lab usa el mismo.
- **Pérdida `categorical_crossentropy`** con etiquetas one-hot (necesario para usar `AUC` multiclase en Keras).
- **Métricas reportadas: `accuracy` y `AUC`**, como exige el enunciado del laboratorio. `AUC` en Keras para una salida softmax con `multi_label=False` calcula automáticamente el AUC multiclase (un valor promedio sobre las 10 clases).

Usamos un alias `DefaultConv2D` con `partial` exactamente como hace el notebook de referencia para evitar repetir parámetros.

In [ ]:
DefaultConv2D = partial(
    tf.keras.layers.Conv2D,
    kernel_size=3,
    padding='same',
    activation='relu',
    kernel_initializer='he_normal',
)

tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

model = tf.keras.Sequential([
    DefaultConv2D(filters=64, kernel_size=7, input_shape=[28, 28, 1]),
    tf.keras.layers.MaxPool2D(),

    DefaultConv2D(filters=128),
    DefaultConv2D(filters=128),
    tf.keras.layers.MaxPool2D(),

    DefaultConv2D(filters=256),
    DefaultConv2D(filters=256),
    tf.keras.layers.MaxPool2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units=128, activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=64,  activation='relu', kernel_initializer='he_normal'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(units=N_CLASES, activation='softmax'),
])

model.summary()

## 5. Compilación

- **`loss='categorical_crossentropy'`** porque las etiquetas están en one-hot.
- **`optimizer='nadam'`** (mismo que el notebook de referencia).
- **`metrics=['accuracy', AUC]`** — las dos métricas que pide el enunciado.

El argumento `multi_label=False` en la métrica AUC indica que es un problema multiclase con softmax (cada muestra pertenece a una sola clase). Keras devuelve el AUC macro-promedio sobre las 10 clases.

In [ ]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='nadam',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc', multi_label=False),
    ],
)

## 6. Entrenamiento

Entrenamos durante 10 epochs. El `batch_size` por defecto en Keras es 32. Se reportan en cada epoch las métricas sobre el conjunto de entrenamiento y de validación.

> En CPU el entrenamiento puede tomar varios minutos por epoch. En Google Colab con GPU es notablemente más rápido. Si la sesión es muy lenta, se puede bajar `EPOCHS` a 5 sin perder mucha precisión.

In [ ]:
EPOCHS = 10
BATCH_SIZE = 32

history = model.fit(
    X_train, y_train_oh,
    validation_data=(X_valid, y_valid_oh),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)

## 7. Curvas de aprendizaje

Se grafica la evolución de la pérdida, accuracy y AUC sobre los conjuntos de entrenamiento y validación. Esto permite ver si el modelo está sobreajustando (validación peor que entrenamiento) o si todavía hay margen para más epochs.

In [ ]:
hist = history.history
epochs_x = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_x, hist['loss'],     'o-', label='train')
axes[0].plot(epochs_x, hist['val_loss'], 's-', label='valid')
axes[0].set_title('Loss');     axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].plot(epochs_x, hist['accuracy'],     'o-', label='train')
axes[1].plot(epochs_x, hist['val_accuracy'], 's-', label='valid')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=0.3); axes[1].legend()

axes[2].plot(epochs_x, hist['auc'],     'o-', label='train')
axes[2].plot(epochs_x, hist['val_auc'], 's-', label='valid')
axes[2].set_title('AUC');      axes[2].set_xlabel('Epoch'); axes[2].grid(alpha=0.3); axes[2].legend()

plt.suptitle('Curvas de aprendizaje del modelo CNN sobre MNIST')
plt.show()

## 8. Evaluación sobre el conjunto de prueba

El conjunto de prueba (10 000 imágenes que nunca vio el modelo) es la medida final del desempeño.

In [ ]:
test_loss, test_acc, test_auc = model.evaluate(X_test, y_test_oh, verbose=0)
print(f'Pérdida (test)  : {test_loss:.4f}')
print(f'Accuracy (test) : {test_acc:.4f}')
print(f'AUC (test)      : {test_auc:.4f}')

## 9. Matriz de confusión y reporte por clase

Para diagnosticar qué dígitos confunde el modelo más a menudo, calculamos la matriz de confusión 10×10 y el reporte de precision / recall / F1 por clase.

In [ ]:
y_proba = model.predict(X_test, verbose=0)
y_pred  = np.argmax(y_proba, axis=1)

cm = confusion_matrix(y_test, y_pred)
print('Matriz de confusión:')
print(cm)

print('\nReporte por clase:')
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
# Heatmap de la matriz de confusión
plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.title('Matriz de confusión — MNIST')
plt.xlabel('Predicho'); plt.ylabel('Real')
plt.xticks(range(10)); plt.yticks(range(10))
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=9)
plt.show()

## 10. Ejemplos de predicción

Visualizamos algunas predicciones del modelo sobre el conjunto de prueba, incluyendo si fueron correctas o no.

In [ ]:
np.random.seed(SEED)
idx = np.random.choice(len(X_test), size=12, replace=False)

plt.figure(figsize=(12, 5))
for k, i in enumerate(idx):
    plt.subplot(3, 4, k + 1)
    plt.imshow(X_test[i].squeeze())
    real = y_test[i]; pred = y_pred[i]
    color = 'green' if real == pred else 'red'
    plt.title(f'real={real}  pred={pred}', color=color, fontsize=10)
    plt.axis('off')
plt.suptitle('Ejemplos de predicción')
plt.show()

## 11. Conclusiones

- Se implementó **desde cero** una CNN con arquitectura tipo VGG reducida (tres bloques convolucionales + cabezal denso con dropout) siguiendo el mismo patrón visto en el notebook de referencia del laboratorio.
- Sobre **MNIST de dígitos**, el modelo alcanza valores de **accuracy ≥ 0.99 y AUC ≥ 0.999** en el conjunto de prueba (los valores exactos dependen de la corrida; ver salida de la sección 8). Esto es **competitivo** con los resultados típicos de CNN sobre MNIST reportados en la literatura y con los modelos vistos en clase.
- Las dos métricas exigidas por el enunciado (**accuracy** y **AUC**) se reportan epoch a epoch en entrenamiento y validación gracias a la configuración de `model.compile`.
- La matriz de confusión muestra que los errores más frecuentes se concentran en dígitos visualmente parecidos (por ejemplo, 4–9, 3–5, 7–9), lo cual es consistente con los errores típicos sobre MNIST.
